# Hierarchical models with JAX!
**Intro:** We have $J$ tumour types and $N_j$ people tested, of which $y_j$ have been diagnosed with cancer. *What is the cancer rate $\theta_j$ estimated from this dataset?* From [this](https://blackjax-devs.github.io/sampling-book/models/change_of_variable_hmc.html).

**The explicit model:** we model the likelihood of the data using a BetaBinomial distribution with parameters $N_j$, $\theta_j$, $\alpha$ and $\beta$, where the last two are shared among the groups (full pooling) and come from an improper prior. The model is:
$$
p(\alpha, \beta) \propto (\alpha + \beta)^{-\frac{5}{2}}
$$
$$
\theta_j \sim \mathrm{Beta}(\alpha, \beta)
$$
$$
y_j \sim \mathrm{Binomial}(N_j, \theta_j)
$$
$$
\text{with} \; \alpha, \beta > 0 \quad \text{and} \quad \theta_j \in \left[0, 1\right]
$$
with $\theta_j$ being partially pooled as there is one value for each group but all groups share the same prior. 

**The compound model:** this model can also be written more concisely with the compound distribution BetaBinomial:
$$
p(\alpha, \beta) \propto (\alpha + \beta)^{-\frac{5}{2}}
$$
$$
y_j \sim \mathrm{BetaBinomial}(N_j, \alpha, \beta)
$$
$$
\text{with} \; \alpha, \beta > 0 \quad \text{and} \quad \theta_j \in \left[0, 1\right]
$$
with $\theta_j$ being marginalised out. To then recover $\theta_j$ we can marginally sample from the posterior:
$$
\theta_j \; |\; N_j, y_j, \alpha, \beta \sim \mathrm{Beta}\left(\alpha + y_j, \beta + N_j - y_j\right)
$$
using the pseudo counts.

**Inference with MCMC:** Infer $\theta_j$ using the compound model, which has fewer parameters. Use NUTS from BlackJAX but first perform a change of variable to remove the constraints of the domain of $\alpha$ and $\beta$ such that the boundaries of the domains don't break the Hamiltonian dynamics simulated via integration.

**Inference with Laplace approximation:** As Beta is the conjugate prior for a Binomial, we can sample $\theta_j$ analytically conditioned on the $y_j$, $\alpha$ and $\beta$ using the pseudo counts. So we can first infer $\alpha$ and $\beta$ from the data (no groups) using MAP and then sample from a Gaussian $\alpha,\,\beta \; |\; y_j, N_j \sim \mathcal{N}\left(\left[\alpha_\text{MAP}, \beta_\text{MAP}\right],\, H_\text{MAP}^{-1}\right)$. We perform a *Laplace approximation*:
1. estimate $\alpha_\text{MAP}, \beta_\text{MAP}$ and $H_\text{MAP}^{-1}$ using grandient descent,
2. approximate $p\left(\alpha, \beta \; |\; y_j, N_j\right) \approx q\left(\alpha, \beta \; |\; y_j, N_j\right)$ with $q\left(\alpha, \beta \; |\; y_j, N_j\right) = \mathcal{N}\left(\left[\alpha_\text{MAP}, \beta_\text{MAP}\right],\, H_\text{MAP}^{-1}\right)$,
3. draw MC samples $\alpha^s, \beta^s \sim q\left(\alpha, \beta \; |\; y_j, N_j\right)$
4. draw MC samples $\theta^s$ from
$$
\theta_j \; |\; N_j, y_j, \alpha^s, \beta^s \sim \mathrm{Beta}\left(\alpha^s + y_j, \beta^s + N_j - y_j\right)
$$

These steps approximate the analytical sampling from the marginal distribution:
$$
p(\theta_j \; | \; y_j, N_j) = \int p\left(\theta_j\;|\; y_j, N_j, \alpha, \beta\right)\, p\left(\alpha, \beta \;|\; y_j, N_j\right)\, d\alpha \, d\beta
$$
with the uncertainty on $\alpha$ and $\beta$ being propagated into $p(\theta_j \; | \; y_j, N_j)$, as if $\alpha$ and $\beta$ vary a lot across samples, the 
$\theta_j$ samples spread out more. 

Note that we need Monte Carlo sampling in step 4 because the marginal integral even with a Gaussian distribution is not solvable in closed-form. If the marginal posterior $p\left(\theta_j\;|\; y_j, N_j, \alpha, \beta\right)$ were to be a Gaussian, then the approximation would have been very effective as the product of two Gaussian pdfs is a Gaussian pdf.

This is a type II inference as described in the prob ML lectures (ADD LINK).

In [ ]:
from functools import partial
from typing import NamedTuple
from datetime import date
import multiprocessing
import random

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import os
import multiprocessing

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(
    multiprocessing.cpu_count()
)

import jax
from jax.scipy import stats
from jax.flatten_util import ravel_pytree
import jax.numpy as jnp

import arviz as az
import blackjax

jax.config.update("jax_enable_x64", True)
rng_key = jax.random.key(int(date.today().strftime("%Y%m%d")))
assert len(jax.devices()) == 8

## The data

In [ ]:
# index of array is type of tumor and value shows number of total people tested.
group_size = jnp.array(
    [
        20,
        20,
        20,
        20,
        20,
        20,
        20,
        19,
        19,
        19,
        19,
        18,
        18,
        17,
        20,
        20,
        20,
        20,
        19,
        19,
        18,
        18,
        25,
        24,
        23,
        20,
        20,
        20,
        20,
        20,
        20,
        10,
        49,
        19,
        46,
        27,
        17,
        49,
        47,
        20,
        20,
        13,
        48,
        50,
        20,
        20,
        20,
        20,
        20,
        20,
        20,
        48,
        19,
        19,
        19,
        22,
        46,
        49,
        20,
        20,
        23,
        19,
        22,
        20,
        20,
        20,
        52,
        46,
        47,
        24,
        14,
    ],
    dtype=jnp.float32,
)

# index of array is type of tumor and value shows number of positve people.
y = jnp.array(
    [
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        2,
        2,
        2,
        2,
        2,
        2,
        2,
        2,
        2,
        1,
        5,
        2,
        5,
        3,
        2,
        7,
        7,
        3,
        3,
        2,
        9,
        10,
        4,
        4,
        4,
        4,
        4,
        4,
        4,
        10,
        4,
        4,
        4,
        5,
        11,
        12,
        5,
        5,
        6,
        5,
        6,
        6,
        6,
        6,
        16,
        15,
        15,
        9,
        4,
    ],
    dtype=jnp.float32,
)

# number of different kind of rat tumors
n_rat_tumors = len(group_size)

_, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].bar(range(n_rat_tumors), y)
axes[0].set_title("No. of positives for each tumor type", fontsize=14)
axes[1].bar(range(n_rat_tumors), group_size)
axes[1].set_xlabel("tumor type", fontsize=12)
axes[1].set_title("Group size for each tumor type", fontsize=14)
plt.tight_layout();

## The model

In [ ]:
# a Python NamedTuple subclass doesn’t need to be registered
# to be considered a pytree node type
class Params(NamedTuple):
    log_a: jnp.ndarray
    log_b: jnp.ndarray


def joint_logdensity(params: Params, y_j, N_j):
    # log_a = log(a) -> a = exp(log_a)
    a, b = jnp.exp(params.log_a), jnp.exp(params.log_b)

    log_prior = -2.5 * jnp.logaddexp(params.log_a, params.log_b)
    loglikelihood = stats.betabinom.logpmf(y_j, N_j, a, b).sum()

    # we are going from p(a, b) to p(log_a, log_b) in the likelihood
    # so we need to mulitply by 1/jac the Jacobian to ensure correct
    # distribution after changing variable. The jac is 1/e^log_a so
    # 1/jac is e^log_a, then take the log (as we are log stuff up)
    return log_prior + loglikelihood + params.log_a + params.log_b

## Inference using Laplace approximation

In [ ]:
def grad_step(params: Params, y_j, N_j, step_size):
    logpost, grad = jax.value_and_grad(joint_logdensity, argnums=0)(params, y_j, N_j)
    updates = jax.tree.map(lambda x, g: x + step_size * g, params, grad)
    return updates, logpost


@jax.jit(static_argnames=["iterations"])
def compute_maxpost_estimate(init_param: Params, y_j, N_j, step_size, iterations):
    # init step fn
    step_fn = partial(grad_step, y_j=y_j, N_j=N_j, step_size=step_size)

    # create wrapper bc scan requires a fn with two args in the signature f: (a, c) -> (a, b)
    def scan_body(params, _):
        new_params, logpost = step_fn(params)
        return new_params, logpost

    return jax.lax.scan(scan_body, init_param, xs=None, length=iterations)


@jax.jit(static_argnames=["posterior_params"])
def mc_samples_params(params: Params, keys, posterior_params):
    # map pytrees to arrays for the inv method and multivariate_normal
    mode = ravel_pytree(params)[0]
    cov_inv = -ravel_pytree(jax.hessian(posterior_params)(params))[0].reshape(2, 2)
    cov = jax.numpy.linalg.inv(cov_inv)  # TODO add some noise for stability

    normal_with_mode_cov = partial(jax.random.multivariate_normal, mean=mode, cov=cov)
    return jax.vmap(normal_with_mode_cov)(keys)

In [ ]:
## LAPLACE APPROXIMATION
# initial conditions and loop hyperparam
params = Params(jnp.log(1.1), jnp.log(2.9))  # now Params is a pytree
iterations, step_size, n_samples = 1000, jnp.array(10 ** (-3)), 1000

# MAP and then print with exp transformed to get back alpha and beta
params_map, logpost_map_iters = compute_maxpost_estimate(
    init_param=params, y_j=y, N_j=group_size, step_size=step_size, iterations=iterations
)
print("Init params:", jax.tree.leaves(jax.tree.map(jnp.exp, params)))
plt.plot(logpost_map_iters)
plt.ylabel("Negative log posterior")
plt.xlabel("Iterations")
print(
    f"MAP params after {iterations} iters:",
    jax.tree.leaves(jax.tree.map(jnp.exp, params_map)),
)

# samples from the q(log_a, log_b) Gaussian approx obtained used Laplace approx
keys = jax.random.split(rng_key, n_samples + 1)
rng_key, keys_a_b_samples = keys[0], keys[1:]

logpost_params = partial(joint_logdensity, y_j=y, N_j=group_size)
samples_params = mc_samples_params(
    params=params_map, keys=keys_a_b_samples, posterior_params=logpost_params
)
samples_a_b = jnp.exp(samples_params)
sns.pairplot(pd.DataFrame(samples_a_b, columns=["a", "b"]))

In [ ]:
## SAMPLE THETA
keys = jax.random.split(rng_key)
rng_key, keys_theta_samples = keys[0], keys[1:].squeeze()
alpha, beta = samples_a_b[:, 0], samples_a_b[:, 1]
# broadcast to get shape SAMPLES x GROUPS
pseudo_counts_pos, pseudo_counts_neg = (
    alpha[:, None] + y[None, :],
    beta[:, None] + group_size[None, :] - y[None, :],
)
thetas = jax.random.beta(
    a=pseudo_counts_pos, b=pseudo_counts_neg, key=keys_theta_samples
)

sns.histplot(
    jnp.ravel(thetas)
)  # this is a bit misleading because we have a distribution for each 71 group

fig, axes = plt.subplots(3, 3, layout="constrained", sharex=True, sharey=True)
for ax, dim in zip(axes.ravel(), random.sample(range(0, thetas.shape[1]), k=axes.size)):
    sns.histplot(thetas[:, dim], ax=ax, binwidth=0.02)
    ax.set_xlabel("Theta posterior")
    ax.set_title(f"Group {dim}", fontdict={"size": 10})
plt.show()

## Inference using multiple chains of NUTS
Based on this [tutorial](https://blackjax-devs.github.io/blackjax/examples/howto_sample_multiple_chains.html), we dont use `vmap` but instead `pmap` to map 8 chains to my 8 logical cores.

In the tutorial, the `vmap` framework works as following:
- run a optimised for loop with `scan` over MCMC iterations
- for each iteration, `vmap` adds a batch axis which is an extra axis that can be used to run the single MCMC iteration in parallel
- XLA gets this extra axis, and decides how to parallelise the computation
We can see that this is not super efficient with NUTS, because in NUTS each MCMC iteration has a variable number of integration steps.
Hence, chains are mostly waiting in the `vmap` for the slowest chain to finish its integration step, thus each MCMC is slow.

Using `pmap` instead, we leverage the available logical cores and send one chain per core.
Each chain does it's thing, indepedentely and in parallel.
`pmap` is less easy to work with:
- need to instruct JAX to split the CPU into multiple devices, do this before any JAX's related imports
```python
import os
import multiprocessing

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(
    multiprocessing.cpu_count()
)
assert len(jax.devices()) == 4  # or your expected nb of logical cores
```
- `pmap` is not able to parallelize the execution when you ask it to perform more computations than there are available deviced, only be able to run as many MCMC chains as you have CPU cores

In [ ]:
def inference_loop(rng_key, kernel, initial_state, num_samples):
    @jax.jit
    def one_step(state, rng_key):
        state, _ = kernel(rng_key, state)
        return state, state

    keys = jax.random.split(rng_key, num_samples)
    _, states = jax.lax.scan(one_step, initial_state, keys)

    return states

In [ ]:
inv_mass_matrix = jnp.ones((2, 2))
step_size = 1e-3
# nb of chains as a multiple of nb of cores
num_chains = 2 * multiprocessing.cpu_count()

initial_positions = Params(jnp.ones(num_chains), jnp.ones(num_chains))
nuts = blackjax.nuts(
    partial(joint_logdensity, y_j=y, N_j=group_size), step_size, inv_mass_matrix
)
initial_states = jax.vmap(nuts.init, in_axes=(0))(initial_positions)

inference_loop_multiple_chains = jax.pmap(
    inference_loop, in_axes=(0, None, 0, None), static_broadcasted_argnums=(1, 3)
)
rng_key, sample_key = jax.random.split(rng_key)
sample_keys = jax.random.split(sample_key, num_chains)

pmap_states = inference_loop_multiple_chains(
    sample_keys, nuts.step, initial_states, 2_000
)